# 4주차: RAG 고도화 (리랭커, 하이브리드) + 기준값 + 멀티턴

3주차 기준선에 기법을 붙여 **같은 질문셋, 같은 채점기**로 비교해요. 결과는 맨 아래 대회 표로 모여요.
코드는 `app/rag_core.py`에 있고, 같은 코드를 `/ask` API(`app/main.py`)도 써요. 그래서 노트북에서 정한 설정(후보 수, 기준값)이 그대로 API 설정이 돼요.

**사전 준비**
1. 저장소 루트에서 `python scripts/merge_wiki.py` (팀 위키 최신화, 페이지 31개)
2. 커널은 3주차와 같은 conda 환경 `study`, 이 폴더에서 `pip install -r requirements.txt`
3. 이 폴더(`members/mingi/week4/`)에 `.env` 만들고 `GEMINI_API_KEY=...` (3주차 `.env` 복사)
4. 첫 실행 때 리랭커 모델 `BAAI/bge-reranker-v2-m3`(약 2GB)를 새로 내려받아요

In [ ]:
import sys, os, json, time
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)
print("GEMINI_API_KEY 불러옴:", bool(os.getenv("GEMINI_API_KEY")))

REPO = Path("../../..").resolve()            # 저장소 루트
WIKI = REPO / "team" / "wiki"
sys.path.append(str(REPO / "team" / "eval"))   # 팀 공통 evaluate.py
sys.path.append(str(Path.cwd()))               # app.rag_core

from evaluate import evaluate, load_questions
from app import rag_core
print(REPO, WIKI.exists())

## 1. 로드 + 분할 (기준선 설정 500/50)

3주차와 같은 코드예요 (`rag_core.load_wiki`, `rag_core.split_docs`).
- 확인: 페이지 수 31개, 청크 수 (3주차 15페이지일 때 109개)

In [ ]:
docs = rag_core.load_wiki(WIKI)
chunks = rag_core.split_docs(docs, chunk_size=500, chunk_overlap=50)
print("페이지 수:", len(docs), "| 청크 수:", len(chunks))
print(chunks[0].metadata)

## 2. 모델 불러오기 + Chroma

임베딩 모델(bi-encoder, `bge-m3`)과 리랭커(cross-encoder, `bge-reranker-v2-m3`)를 한 번만 불러와요.

In [ ]:
embeddings, cross_encoder = rag_core.load_models()
store = rag_core.build_vectorstore(chunks, embeddings, collection_name="wiki-week4")
print("저장된 청크 수:", store._collection.count())
print(cross_encoder)

## 3. 새 기준선 (페이지 31개)

질문셋 두 가지로 재요.
- **질문 25개**: 3주차와 같은 질문 (1~2주차 페이지만 정답). 3주차 점수(Recall 1.000, MRR 0.823)와 비교용
- **질문 38개**: 25개 + 3주차 페이지용 초안 13개 (`4주차_질문셋_초안.jsonl`). 팀에서 OK하면 이게 대회 기준이에요

초안이 이미 `team/eval/questions.jsonl`에 들어가 있으면 중복 없이 합쳐요.

In [ ]:
K = 5
qs_team = load_questions(REPO / "team" / "eval" / "questions.jsonl")
qs_draft = load_questions(Path.cwd() / "4주차_질문셋_초안.jsonl")
team_ids = {q["id"] for q in qs_team}
QS_25 = [q for q in qs_team if q["id"] not in {d["id"] for d in qs_draft}][:25]
QS_ALL = qs_team + [q for q in qs_draft if q["id"] not in team_ids]
print("질문 25개 세트:", len(QS_25), "| 전체 세트:", len(QS_ALL))

baseline = store.as_retriever(search_kwargs={"k": K})
base_25 = evaluate(baseline, QS_25, k=K)
base_all = evaluate(baseline, QS_ALL, k=K, show_misses=True)

In [ ]:
# 기준선 결과를 파일로 남겨요 (README, 위키 측정 결과용)
(Path.cwd() / "baseline.json").write_text(json.dumps(
    {"config": {"chunk_size": 500, "overlap": 50, "k": K, "pages": len(docs)},
     "q25": {k: base_25[k] for k in ("recall", "mrr", "avg_ms")},
     "all": base_all}, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장: baseline.json")

**📝 관찰 메모 (3. 새 기준선)**
- 페이지가 15개에서 31개로 늘었을 때 질문 25개 점수는 어떻게 바뀌었나? 왜?
- 1위를 놓친 질문에서 1위를 뺏은 페이지는 주로 어떤 페이지였나? (같은 주제를 다른 사람이 쓴 페이지인가?)
- Recall이 떨어졌나, MRR만 떨어졌나? 그럼 리랭커와 하이브리드 중 어느 쪽이 맞을까?
-

## 4. 리랭커: 질문 하나로 먼저 보기

벡터 검색으로 후보 20개를 뽑고, 크로스인코더가 (질문, 청크) 쌍을 하나씩 읽어 다시 채점해 상위 5개만 남겨요.
기준선에서 1위를 놓친 질문 하나로 두 결과를 나란히 봐요.

In [ ]:
reranker = rag_core.RerankRetriever(vectorstore=store, cross_encoder=cross_encoder, fetch_k=20, top_n=K)

miss = next((r for r in base_all["rows"] if r["rank"] != 1), base_all["rows"][0])
q = miss["question"]
print("Q.", q, "| 정답:", miss["answer_pages"], "\n")

rows = []
vec_docs = store.similarity_search_with_score(q, k=K)
rr_docs = reranker.invoke(q)
for i in range(K):
    vd, vs = vec_docs[i]
    rows.append({"순위": i + 1,
                 "기준선 청크": vd.metadata["id"], "코사인 거리": round(vs, 4),
                 "리랭커 청크": rr_docs[i].metadata["id"], "리랭커 점수": round(rr_docs[i].metadata["rerank_score"], 4)})
pd.DataFrame(rows)

**📝 관찰 메모 (4. 리랭커 하나 보기)**
- 리랭커 점수는 어떤 범위의 숫자인가? 코사인 거리와 방향이 같은가 반대인가? (클수록 가까운가?)
- 정답 페이지가 몇 위에서 몇 위로 움직였나?
- bi-encoder와 cross-encoder는 질문과 청크를 어떻게 다르게 읽나? 왜 처음부터 cross-encoder로 전체를 검색하지 않나?
-

## 5. 리랭커 채점 + 후보 수 비교

후보 수(fetch_k)를 10, 20, 30으로 바꿔요. 후보가 많을수록 정답이 후보에 들어올 확률은 오르지만, 크로스인코더가 읽을 쌍이 늘어서 느려져요.

In [ ]:
rerank_results = {}
table = [{"구성": "기준선 (벡터 k=5)", "Recall@5": round(base_all["recall"], 3),
          "MRR@5": round(base_all["mrr"], 3), "질문당 ms": round(base_all["avg_ms"], 1)}]
for fk in (10, 20, 30):
    r = rag_core.RerankRetriever(vectorstore=store, cross_encoder=cross_encoder, fetch_k=fk, top_n=K)
    r.invoke("워밍업")      # 첫 호출은 느리니 시간에서 빼요
    res = evaluate(r, QS_ALL, k=K, verbose=False)
    rerank_results[fk] = res
    table.append({"구성": f"+ 리랭커 (후보 {fk} -> 5)", "Recall@5": round(res["recall"], 3),
                  "MRR@5": round(res["mrr"], 3), "질문당 ms": round(res["avg_ms"], 1)})
pd.DataFrame(table)

In [ ]:
# 기준선 대비 질문별 순위 변화 (후보 20 기준)
def rank_changes(before, after):
    b = {r["id"]: r["rank"] for r in before["rows"]}
    out = []
    for r in after["rows"]:
        if b[r["id"]] != r["rank"]:
            out.append({"id": r["id"], "기준선 순위": b[r["id"]], "새 순위": r["rank"], "질문": r["question"][:40]})
    return pd.DataFrame(out)

rank_changes(base_all, rerank_results[20])

**📝 관찰 메모 (5. 리랭커 채점)**
- MRR은 얼마나 올랐고, 질문당 시간은 얼마나 늘었나? 채택 규칙으로 보면 채택인가?
- 기준선에서 1위를 놓친 질문 중 몇 개가 1위로 올라왔나? 반대로 떨어진 질문은? 왜 떨어졌을까?
- 후보 수 10/20/30 중 무엇을 고르겠나? 이유는?
-

## 6. (비교용) 하이브리드: BM25 + 벡터, RRF

리랭커와 비교하려고 하이브리드도 한 줄 재 봐요. BM25는 단어가 겹치는지로, 벡터는 뜻이 비슷한지로 순위를 매기고, RRF는 두 순위를 `1 / (60 + 순위)`로 더해 합쳐요.
토큰화는 띄어쓰기 수준이라 한국어 조사("컨테이너가", "컨테이너를")를 구분 못 하는 한계가 있어요.

In [ ]:
hybrid = rag_core.HybridRetriever(vectorstore=store, chunks=chunks, fetch_k=20, top_n=K)
hyb_res = evaluate(hybrid, QS_ALL, k=K, show_misses=True)
print(rag_core.simple_tokenize("컨테이너가 꺼진 뒤에도 docker run -v 옵션으로 결과를 남기려면?"))

**📝 관찰 메모 (6. 하이브리드)**
- 하이브리드는 기준선보다 Recall과 MRR 중 무엇을 바꿨나? 리랭커와 비교하면?
- BM25가 특히 잘 찾은 질문이 있었나? (명령어, 숫자, 고유명사가 들어간 질문)
- 토큰 출력에서 한국어 토큰화의 한계가 보이나?
-

## 7. 위키에 없는 질문 거르기: 기준값(MIN_SCORE) 정하기

3주차에는 위키에 없는 질문에도 출처 4개를 돌려줬어요. 리랭커 1위 점수가 기준값보다 낮으면 LLM을 부르지 않고 "위키에 없는 내용"으로 답하게 할 거예요.
위키에 있는 질문(질문셋 전체)과 없는 질문의 1위 점수를 나란히 보고 그 사이에서 기준값을 골라요.

In [ ]:
OUT_OF_WIKI = [
    "쿠버네티스 파드란 무엇인가?",
    "LoRA 파인튜닝에서 rank는 무엇을 뜻해?",
    "AWS S3 버킷을 만드는 방법은?",
    "React에서 useState는 언제 써?",
    "오늘 점심 메뉴 추천해 줘",
    "GitHub Actions로 테스트를 자동 실행하려면?",
]
reranker = rag_core.RerankRetriever(vectorstore=store, cross_encoder=cross_encoder, fetch_k=20, top_n=K)
in_scores = [reranker.invoke(q["question"])[0].metadata["rerank_score"] for q in QS_ALL]
out_scores = [reranker.invoke(q)[0].metadata["rerank_score"] for q in OUT_OF_WIKI]

print(f"위키에 있는 질문 {len(in_scores)}개: 최소 {min(in_scores):.4f}, 중간값 {sorted(in_scores)[len(in_scores)//2]:.4f}")
print(f"위키에 없는 질문 {len(out_scores)}개: 최대 {max(out_scores):.4f}")
pd.DataFrame({"위키에 없는 질문": OUT_OF_WIKI, "1위 점수": [round(s, 4) for s in out_scores]})

In [ ]:
# 기준값 후보별로: 있는 질문을 잘못 거절한 수 / 없는 질문을 통과시킨 수
rows = []
for th in (0.01, 0.05, 0.1, 0.2, 0.3, 0.5):
    rows.append({"MIN_SCORE": th,
                 "있는 질문 잘못 거절": sum(s < th for s in in_scores),
                 "없는 질문 통과": sum(s >= th for s in out_scores)})
pd.DataFrame(rows)

**📝 관찰 메모 (7. 기준값)**
- 있는 질문과 없는 질문의 점수가 깔끔하게 나뉘었나? 겹친다면 어떤 질문이 겹쳤나?
- 기준값을 몇으로 정했나? 잘못 거절하는 것과 잘못 통과시키는 것 중 무엇이 더 나쁜가?
- (정한 값은 `.env`의 `MIN_SCORE`에 적어요)
-

## 8. 선택 항목: 멀티턴 (후속 질문 다시 쓰기)

"그럼 compose에서는?"처럼 앞 대화를 봐야 뜻이 통하는 질문은 그대로 검색하면 엉뚱한 문서가 나와요.
LLM으로 먼저 독립된 질문으로 바꾼 뒤 검색해요. `/ask`의 `history` 필드가 이 기능이에요.

In [ ]:
llm = rag_core.make_llm(os.getenv("GEMINI_API_KEY"))
history = [
    {"role": "user", "content": "docker run에서 컨테이너가 꺼져도 파일을 남기려면 어떻게 해?"},
    {"role": "assistant", "content": "-v 옵션으로 볼륨을 연결하면 컨테이너가 꺼져도 파일이 남아요."},
]
follow_up = "그럼 compose에서는 어떻게 써?"

rewritten = rag_core.condense_question(llm, follow_up, history)
print("원래 질문:  ", follow_up)
print("다시 쓴 질문:", rewritten)

for name, query in [("그대로 검색", follow_up), ("다시 쓴 질문으로 검색", rewritten)]:
    print(f"\n--- {name} ---")
    for d in reranker.invoke(query):
        print(f"{d.metadata['rerank_score']:.4f}  {d.metadata['id']}")

In [ ]:
# /ask와 똑같은 함수로 끝까지: 답 + 출처 (MIN_SCORE는 7번에서 정한 값으로)
MIN_SCORE = 0.1
for question, hist in [("docker compose에서 api 컨테이너가 Redis에 접속할 때 주소를 어떻게 써야 해?", []),
                       (follow_up, history),
                       ("쿠버네티스 파드란 무엇인가?", [])]:
    out = rag_core.answer(llm, reranker, question, history=hist, min_score=MIN_SCORE)
    print(f"Q. {question}\n  검색어: {out['search_query']}\n  1위 점수: {out['top_score']:.4f}  위키에 있음: {out['in_wiki']}")
    print("  답:", out["answer"][:200].replace("\n", " "))
    print("  출처:", [s["page"] for s in out["sources"]], "\n")

**📝 관찰 메모 (8. 멀티턴)**
- 다시 쓴 질문은 어떻게 바뀌었나? 검색 결과가 얼마나 달라졌나?
- 질문마다 LLM 호출이 한 번 더 늘어요. 기록이 없을 때는 호출하지 않게 한 이유는?
- 위키에 없는 질문이 이제 출처 없이 거절되나?
-

## 9. 대회 표

단체 세션에 들고 갈 표예요. README와 위키 `hybrid-rerank.md`의 측정 결과에도 그대로 써요.

In [ ]:
contest = [
    {"구성": "기준선 (bge-m3, 500/50, k=5)", "질문 수": len(QS_ALL), **{m: round(base_all[k], 3) for m, k in [("Recall@5", "recall"), ("MRR@5", "mrr")]}, "질문당 ms": round(base_all["avg_ms"], 1)},
    {"구성": "+ 리랭커 (후보 20 -> 5)", "질문 수": len(QS_ALL), **{m: round(rerank_results[20][k], 3) for m, k in [("Recall@5", "recall"), ("MRR@5", "mrr")]}, "질문당 ms": round(rerank_results[20]["avg_ms"], 1)},
    {"구성": "+ 하이브리드 (BM25 + 벡터, RRF)", "질문 수": len(QS_ALL), **{m: round(hyb_res[k], 3) for m, k in [("Recall@5", "recall"), ("MRR@5", "mrr")]}, "질문당 ms": round(hyb_res["avg_ms"], 1)},
    {"구성": "참고: 기준선, 질문 25개", "질문 수": len(QS_25), **{m: round(base_25[k], 3) for m, k in [("Recall@5", "recall"), ("MRR@5", "mrr")]}, "질문당 ms": round(base_25["avg_ms"], 1)},
]
df = pd.DataFrame(contest)
(Path.cwd() / "contest.json").write_text(df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
# README에 그대로 붙여 넣을 마크다운 표
print("| " + " | ".join(df.columns) + " |")
print("| " + " | ".join("---" for _ in df.columns) + " |")
for row in df.itertuples(index=False):
    print("| " + " | ".join(str(v) for v in row) + " |")
df

**📝 관찰 메모 (9. 대회 표)**
- 팀 프로젝트에 넣을 구성은? 채택 규칙(점수 상승 + 감수할 만한 지연)으로 한 줄 근거
- 이 점수를 얼마나 믿을 수 있나? (질문 38개, 페이지 31개, 질문을 누가 만들었나)
-